In [1]:
%load_ext autoreload
%autoreload 2
# %matplotlib widget
%pdb off

from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, HTML
import pyafn
from pyafn import rho, Cd
import random
from scipy.stats import lognorm
from scipy.stats import norm
from scipy.optimize import curve_fit
from emulationHelpers import readEmulationMI, plot_ventilation_model_fit

#close all figures
plt.close('all')
plt.rcParams['figure.dpi'] = 140
im_scaling = .75
plt.rcParams['figure.figsize'] = [6.4 * im_scaling, 4.8 * im_scaling]
plt.rcParams['font.family'] = 'Times New Roman'

home_dir = "./"
display(home_dir)
plt.close('all')

Automatic pdb calling has been turned OFF


'./'

In [2]:
flowStatsMI, roomVentilationMI = readEmulationMI(home_dir=home_dir)

y_var = "flux-Norm"
x_var = "p-noInt_optp0-q_model-Norm"

Normalizing p cols: ['EP_p_avg', 'EP_p_avg-noInt', 'p-noInt_optp0-p0', 'p-noInt_optp0Cd-p0', 'pEP-noInt_optp0-p0', 'pEP-noInt_optp0Cd-p0', 'pEP_optp0-p0', 'pEP_optp0Cd-p0', 'p_avg-noInt', 'p_intensity-noInt', 'p_rms-noInt']
Normalizing u cols: ['EPR_mag', 'EPR_mag-noInt', 'EP_comp(u_avg,0)', 'EP_comp(u_avg,0)-noInt', 'EP_comp(u_avg,1)', 'EP_comp(u_avg,1)-noInt', 'EP_comp(u_avg,2)', 'EP_comp(u_avg,2)-noInt', 'EP_mag', 'EP_mag(u)_avg', 'EP_mag(u)_avg-noInt', 'EP_mag-noInt', 'EP_normal', 'EP_normal-noInt', 'EP_shear', 'EP_shear-noInt', 'EP_shear-noInt-qIn', 'EP_shear-noInt-qOut', 'EP_shear_o_qmodel', 'comp(u_avg,0)-noInt', 'comp(u_avg,1)-noInt', 'comp(u_avg,2)-noInt', 'flux', 'p-noInt_optp0-netq_model', 'p-noInt_optp0-q_model', 'p-noInt_optp0Cd-netq_model', 'p-noInt_optp0Cd-q_model', 'pEP-noInt_optp0-netq_model', 'pEP-noInt_optp0-q_model', 'pEP-noInt_optp0Cd-netq_model', 'pEP-noInt_optp0Cd-q_model', 'pEP_optp0-netq_model', 'pEP_optp0-q_model', 'pEP_optp0Cd-netq_model', 'pEP_optp0Cd-q_mode

/oak/stanford/groups/gorle/nbachand/Cascade/city_block_cfd/emulationHelpers.py:82: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  roomVentilationMI["flux-Norm"] = roomVentilationMI["flux"] / roomVentilationMI["WS"]
/oak/stanford/groups/gorle/nbachand/Cascade/city_block_cfd/emulationHelpers.py:83: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  roomVentilationMI["p-noInt_optp0-q_model-Norm"] = roomVentilationMI["p-noInt_optp0-q_model"] / roomVentilationMI["WS"]


In [3]:
import pymc as pm

from emulationHelpers import ventilation_redecomp_p_op

seed = 42
rng = np.random.default_rng(seed)
n_trials = 100
u_model_data = np.ones(n_trials, dtype=float)

true_a = 1.25
true_p_rms = 0.03
true_sigma = 0.02

mu_true = np.asarray(
    pyafn.ventilationReDecomp_p(u_model_data, true_a, true_p_rms),
    dtype=float,
)
y_obs = rng.normal(mu_true, true_sigma)

coords = {"trial": np.arange(n_trials)}

with pm.Model(coords=coords) as ventilation_model:
    u_model = pm.Data("u_model", u_model_data, dims="trial")

    a = pm.HalfNormal("a", sigma=2.0)
    p_rms = pm.HalfNormal("p_rms", sigma=0.1)
    sigma = pm.HalfNormal("sigma", sigma=0.1)

    # Slice works well here because the wrapped NumPy pyafn call is not differentiable to PyMC.
    mu = pm.Deterministic(
        "mu",
        ventilation_redecomp_p_op(u_model, a, p_rms),
        dims="trial",
    )

    flux_obs = pm.Normal("flux_obs", mu=mu, sigma=sigma, observed=y_obs, dims="trial")

    idata = pm.sample(
        draws=2000,
        tune=2000,
        chains=4,
        cores=4,
        step=pm.Slice(),
        random_seed=seed,
    )

summary = pm.stats.summary(idata, var_names=["a", "p_rms", "sigma"])
display(summary)
print(f"True values: a={true_a:.4f}, p_rms={true_p_rms:.4f}, sigma={true_sigma:.4f}")


Multiprocess sampling (4 chains in 4 jobs)
CompoundStep
>Slice: [a]
>Slice: [p_rms]
>Slice: [sigma]


Output()

Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 20 seconds.


,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
a,1.249,0.002,1.246,1.253,0.000,0.000,5680.0,4119.0,1.0
p_rms,0.080,0.060,0.000,0.187,0.001,0.001,4374.0,4111.0,1.0
sigma,0.016,0.001,0.014,0.018,0.000,0.000,7276.0,5257.0,1.0


True values: a=1.2500, p_rms=0.0300, sigma=0.0200
